## Baseline：阈值 q 扫描 + 后处理网格搜索（不重训）\n\n### 用法\n1) 先在 `baseline_transformer_postprocess.ipynb`（或任何能得到分数的 notebook）跑出：\n- `train_point_scores`（训练集每个时间点的分数）\n- `test_point_scores`（测试集每个时间点的分数）\n- `test_labels`（测试集点级标签，0/1）\n\n2) 在那个 notebook 里，把这三个变量复制到本 notebook 的 **输入 cell**，然后运行后面的搜索。\n\n### 输出\n- Top-10 by point-wise F1\n- Top-10 by event-level PA_F1（point-adjust）\n

In [ ]:
# ============ 输入：把你的三个变量粘贴/赋值到这里 ============
# train_point_scores = ...  # shape [train_len]
# test_point_scores  = ...  # shape [test_len]
# test_labels        = ...  # shape [test_len], 0/1

import numpy as np
import pandas as pd

assert 'train_point_scores' in globals(), '请先定义 train_point_scores'
assert 'test_point_scores' in globals(),  '请先定义 test_point_scores'
assert 'test_labels' in globals(),        '请先定义 test_labels'

train_point_scores = np.asarray(train_point_scores, dtype=np.float64)
test_point_scores  = np.asarray(test_point_scores, dtype=np.float64)
test_labels        = np.asarray(test_labels, dtype=int)

print('train_point_scores:', train_point_scores.shape)
print('test_point_scores :', test_point_scores.shape)
print('test_labels       :', test_labels.shape, 'anomaly=', int(test_labels.sum()))


In [ ]:
# ============ 指标 + 后处理工具 ============
from sklearn.metrics import precision_recall_fscore_support

def prf1(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='binary', zero_division=0
    )
    return float(p), float(r), float(f1)

def point_adjust(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """对每个真实异常连续段，只要命中任意一点，就把整段预测置为 1。"""
    y_true = y_true.astype(int)
    y_pred = y_pred.astype(int).copy()
    n = len(y_true)
    i = 0
    while i < n:
        if y_true[i] == 1:
            j = i
            while j < n and y_true[j] == 1:
                j += 1
            if y_pred[i:j].max() == 1:
                y_pred[i:j] = 1
            i = j
        else:
            i += 1
    return y_pred

def moving_average(x: np.ndarray, k: int) -> np.ndarray:
    if k <= 1:
        return x
    k = int(k)
    pad = k // 2
    xp = np.pad(x, (pad, pad), mode='edge')
    kernel = np.ones(k, dtype=np.float64) / k
    return np.convolve(xp, kernel, mode='valid')

def apply_segment_constraints(pred: np.ndarray, min_len: int = 10, merge_gap: int = 5) -> np.ndarray:
    """去毛刺：短于 min_len 的异常段删除；间隔 <= merge_gap 的异常段合并。"""
    pred = pred.astype(int)
    n = len(pred)
    segments = []
    i = 0
    while i < n:
        if pred[i] == 1:
            j = i
            while j < n and pred[j] == 1:
                j += 1
            segments.append([i, j])
            i = j
        else:
            i += 1

    merged = []
    for s, e in segments:
        if not merged:
            merged.append([s, e])
        else:
            ps, pe = merged[-1]
            if s - pe <= merge_gap:
                merged[-1][1] = e
            else:
                merged.append([s, e])

    out = np.zeros(n, dtype=int)
    for s, e in merged:
        if (e - s) >= int(min_len):
            out[s:e] = 1
    return out


In [ ]:
# ============ 搜索配置（你可以先用默认小网格，跑得快） ============
q_list = np.unique(np.concatenate([
    np.linspace(0.90, 0.97, 15),
    np.linspace(0.971, 0.995, 13),
    np.linspace(0.996, 0.9995, 8),
]).round(6))

smooth_k_list = [1, 5, 11, 21, 31]  # 1 表示不平滑
min_len_list = [1, 5, 10, 20, 40]
merge_gap_list = [0, 2, 5, 10, 20]

print('q candidates:', len(q_list))
print('grid size:', len(smooth_k_list) * len(min_len_list) * len(merge_gap_list))


In [ ]:
# ============ 主搜索：q + (smooth, min_len, merge_gap) ============
rows = []
test_scores_f64 = test_point_scores.astype(np.float64)

for q in q_list:
    th = float(np.quantile(train_point_scores, q))

    # raw
    raw_pred = (test_scores_f64 > th).astype(int)
    raw_p, raw_r, raw_f1 = prf1(test_labels, raw_pred)
    raw_pa_pred = point_adjust(test_labels, raw_pred)
    raw_pa_p, raw_pa_r, raw_pa_f1 = prf1(test_labels, raw_pa_pred)
    rows.append({
        'mode': 'raw', 'q': float(q), 'th': th,
        'smooth_k': None, 'min_len': None, 'merge_gap': None,
        'P': raw_p, 'R': raw_r, 'F1': raw_f1,
        'PA_P': raw_pa_p, 'PA_R': raw_pa_r, 'PA_F1': raw_pa_f1,
    })

    # post
    for smooth_k in smooth_k_list:
        sm_scores = moving_average(test_scores_f64, smooth_k)
        base_pred = (sm_scores > th).astype(int)
        for min_len in min_len_list:
            for merge_gap in merge_gap_list:
                post_pred = apply_segment_constraints(base_pred, min_len=min_len, merge_gap=merge_gap)
                p, r, f1 = prf1(test_labels, post_pred)
                post_pa_pred = point_adjust(test_labels, post_pred)
                pa_p, pa_r, pa_f1 = prf1(test_labels, post_pa_pred)
                rows.append({
                    'mode': 'post', 'q': float(q), 'th': th,
                    'smooth_k': int(smooth_k), 'min_len': int(min_len), 'merge_gap': int(merge_gap),
                    'P': p, 'R': r, 'F1': f1,
                    'PA_P': pa_p, 'PA_R': pa_r, 'PA_F1': pa_f1,
                })

df = pd.DataFrame(rows)

print('Top-10 by point-wise F1')
display(df.sort_values(['F1', 'PA_F1'], ascending=[False, False]).head(10))

print('Top-10 by event-level PA_F1')
display(df.sort_values(['PA_F1', 'F1'], ascending=[False, False]).head(10))
